# Tema 9 · Combinación de modelos: bagging, random forest y boosting

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

La **Viña Altos de Quillón** produce uva País y Cinsault en el Valle del Itata. Cada temporada pierde parte de la cosecha por **botritis**, un hongo que pudre los racimos cuando hay humedad. Si supiera qué cuarteles (paños de la viña) se van a enfermar, podría aplicar fungicida a tiempo o **adelantar la cosecha** solo en esos. El enólogo probó un árbol de decisión, pero cada vez que agregaba datos el árbol cambiaba por completo y no sabía en cuál confiar.

> **❓ La pregunta del cliente:** ¿Qué modelo conviene usar para anticipar en qué cuarteles aparecerá la botritis, y cuánto se gana combinando muchos árboles en vez de usar uno solo?

## 🗂️ Los datos: `t9_botritis.csv` (800 cuarteles con 9 atributos de clima, follaje y manejo, y si tuvieron botritis)

| Columna | Qué significa |
|---|---|
| `cuartel` | número del registro (cuartel y semana) |
| `humedad_relativa_pct` | humedad relativa media de la semana, en % |
| `dias_lluvia` | días con lluvia en las últimas dos semanas |
| `temperatura_media_c` | temperatura media de la semana, en °C |
| `densidad_follaje` | densidad del follaje, de 1 (ralo) a 5 (muy denso) |
| `dias_desde_fungicida` | días desde la última aplicación de fungicida |
| `pendiente_pct` | pendiente del cuartel, en % |
| `altitud_m` | altitud, en metros |
| `edad_parras` | edad de las parras, en años |
| `distancia_bodega_km` | distancia del cuartel a la bodega, en km |
| `botritis` | **si el cuartel tuvo botritis (Sí/No)** ← lo que se quiere predecir |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t9_botritis.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 800 filas y 11 columnas.

## 1. Conocer los datos

📘 Cada fila es un **cuartel** (un paño de la viña) en una semana de pre-cosecha. Hay **9 atributos** medidos en el
cuartel (clima, follaje, manejo) y la columna **`botritis`**, que dice si el cuartel terminó con el hongo. Es una
**clasificación**: el modelo debe decir «Sí» o «No» para cuarteles nuevos. La celda cuenta los casos de cada clase.

In [ ]:
datos.head()

In [ ]:
X = datos.drop(columns=["cuartel", "botritis"])   # los 9 atributos
y = datos["botritis"]                              # lo que se quiere predecir
print("Cuarteles:", len(datos), "· Atributos:", X.shape[1])
print("Con botritis:", (y == "Sí").sum(), "· Sin botritis:", (y == "No").sum())
# ✅ Debe mostrar: Cuarteles: 800 · Atributos: 9  /  Con botritis: 184 · Sin botritis: 616

## 2. Separar entrenamiento y prueba

📘 El 70 % de los cuarteles se usa para **entrenar** los modelos y el 30 % para **probarlos** con casos que no vieron.
`stratify=y` mantiene la misma proporción de cuarteles con botritis en las dos partes. Como referencia se calcula
**ZeroR**: decir siempre «No» (la clase más común).

In [ ]:
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.3, random_state=2026, stratify=y)
print("Entrenamiento:", len(X_ent), "· Prueba:", len(X_pru))
print(f"ZeroR (decir siempre «No») acierta el {(y_pru == 'No').mean():.1%} de los cuarteles de prueba")
# ✅ Debe mostrar: Entrenamiento: 560 · Prueba: 240  /  ZeroR (decir siempre «No») acierta el 77.1% de los cuarteles de prueba

## 3. Un solo árbol de decisión

📘 Se entrena el árbol que ya conocen del curso, **sin límite de profundidad**: puede seguir dividiendo hasta que
cada hoja tenga cuarteles de una sola clase.

In [ ]:
arbol = DecisionTreeClassifier(random_state=0).fit(X_ent, y_ent)
print(f"Árbol → entrenamiento: {arbol.score(X_ent, y_ent):.1%} · prueba: {arbol.score(X_pru, y_pru):.1%}")
# ✅ Debe mostrar: Árbol → entrenamiento: 100.0% · prueba: 82.5%

### ✍️ ¿Por qué el árbol acierta el 100 % en entrenamiento y bastante menos en prueba? ¿Qué nombre tiene eso?

_Su respuesta:_

## 4. El problema: un árbol es inestable

📘 Se imita lo que pasaría si la viña hubiera juntado **otra muestra** de cuarteles. Se arman **5 muestras
con reemplazo** del entrenamiento (cada una del mismo tamaño, pero con algunos cuarteles repetidos y otros fuera) y se
entrena un árbol con cada una. Después se cuenta en cuántos cuarteles de prueba **los 5 árboles no están de acuerdo**.

In [ ]:
generador = np.random.default_rng(0)
votos = []
for i in range(5):
    muestra = generador.integers(0, len(X_ent), len(X_ent))          # cuarteles elegidos con reemplazo
    arbol_i = DecisionTreeClassifier(random_state=0).fit(X_ent.iloc[muestra], y_ent.iloc[muestra])
    votos.append(arbol_i.predict(X_pru))
    print(f"Árbol {i + 1}: acierta el {(votos[-1] == y_pru).mean():.1%} de los cuarteles de prueba")

votos = np.array(votos)
desacuerdo = (~(votos == votos[0]).all(axis=0)).sum()
print(f"Cuarteles de prueba en que los 5 árboles NO coinciden: {desacuerdo} de {len(X_pru)}")
# ✅ Debe mostrar: Árbol 1: acierta el 80.8% de los cuarteles de prueba  /  Árbol 2: acierta el 79.2% de los cuarteles de prueba  /  Árbol 3: acierta el 74.2% de los cuarteles de prueba  /  Árbol 4: acierta el 78.3% de los cuarteles de prueba  /  Árbol 5: acierta el 82.1% de los cuarteles de prueba  /  Cuarteles de prueba en que los 5 árboles NO coinciden: 105 de 240

## 5. La solución: que voten

📘 En vez de elegir un árbol, se les pide que **voten**: para cada cuartel gana la respuesta de la mayoría (3 o más
de 5). Esto es **bagging** hecho a mano: *b*ootstrap (muestras con reemplazo) + *agg*regating (juntar los votos).

In [ ]:
voto_mayoria = np.where((votos == "Sí").sum(axis=0) >= 3, "Sí", "No")
print(f"Promedio de los 5 árboles solos: {np.mean([(v == y_pru).mean() for v in votos]):.1%}")
print(f"Votación de los 5 árboles:       {(voto_mayoria == y_pru).mean():.1%}")
# ✅ Debe mostrar: Promedio de los 5 árboles solos: 78.9%  /  Votación de los 5 árboles:       81.7%

### ✍️ ¿Qué tan distintos fueron los 5 árboles? ¿Por qué la votación acierta más que un árbol típico, si los 5 aprendieron de los mismos datos?

_Su respuesta:_

## 6. Los métodos de combinación de scikit-learn

📘 Se comparan cinco modelos con los mismos datos:
- **Un árbol** sin límite y uno de **profundidad 4** (podado).
- **Bagging**: 100 árboles, cada uno con su muestra con reemplazo, que votan.
- **Random forest**: como bagging, pero además cada división del árbol solo puede elegir entre **algunas columnas al
  azar**. Así los árboles quedan más distintos entre sí y la votación funciona mejor.
- **Boosting** (`GradientBoostingClassifier`): 100 árboles pequeños **en cadena**; cada uno se dedica a corregir los
  errores que dejaron los anteriores.

In [ ]:
modelos = {
    "Un árbol (sin límite)": DecisionTreeClassifier(random_state=0),
    "Un árbol (profundidad 4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "Bagging (100 árboles)": BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=0),
    "Random forest (100 árboles)": RandomForestClassifier(n_estimators=100, random_state=0),
    "Boosting (100 árboles)": GradientBoostingClassifier(random_state=0),
}
for nombre, modelo in modelos.items():
    modelo.fit(X_ent, y_ent)
    print(f"{nombre:<28} entrenamiento: {modelo.score(X_ent, y_ent):.1%} · prueba: {modelo.score(X_pru, y_pru):.1%}")
# ✅ Debe mostrar: Un árbol (sin límite)        entrenamiento: 100.0% · prueba: 82.5%  /  Un árbol (profundidad 4)     entrenamiento: 88.0% · prueba: 82.9%  /  Bagging (100 árboles)        entrenamiento: 100.0% · prueba: 86.2%  /  Random forest (100 árboles)  entrenamiento: 100.0% · prueba: 87.5%  /  Boosting (100 árboles)       entrenamiento: 100.0% · prueba: 89.6%

### ✍️ ¿Qué modelo acierta más en prueba? ¿Cuántos puntos gana sobre el árbol solo? Si los modelos combinados también aciertan cerca del 100 % en entrenamiento, ¿por qué no sobreajustan igual que el árbol?

_Su respuesta:_

## 7. ¿Cuántos árboles hacen falta?

📘 Se entrena el random forest con distinto número de árboles (`n_estimators`) y se mide su acierto en prueba.

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
N_ARBOLES = 100        # ✏️ pruebe con 1, 5, 10, 25, 50, 100 y 200 (al terminar, deje 100)

bosque = RandomForestClassifier(n_estimators=N_ARBOLES, random_state=0).fit(X_ent, y_ent)
print(f"Random forest con {N_ARBOLES} árboles → acierto en prueba: {bosque.score(X_pru, y_pru):.1%}")

**✏️ Anoten aquí sus resultados:**

| Número de árboles | Acierto en prueba |
|:-:|:-:|
| 1 | |
| 5 | |
| 10 | |
| 25 | |
| 50 | |
| 100 | |
| 200 | |

### ✍️ ¿Cómo cambia el acierto al agregar árboles? ¿Desde cuántos árboles casi no mejora? ¿Hay algún costo en usar muchos?

_Su respuesta:_

## 8. ¿Qué mira el bosque? Importancia de los atributos

📘 Un bosque de 100 árboles no se puede dibujar como un árbol, pero sí se puede medir **cuánto usó cada atributo**
para separar los cuarteles con y sin botritis (`feature_importances_`; las importancias suman 1).

In [ ]:
bosque = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_ent, y_ent)
importancia = pd.Series(bosque.feature_importances_, index=X_ent.columns).sort_values()

plt.figure(figsize=(7, 4))
plt.barh(importancia.index, importancia.values, color="#6b2d5c")
plt.xlabel("Importancia en el random forest")
plt.title("¿Qué atributos usa el bosque para anticipar la botritis?")
plt.show()

print("Los 5 más importantes:", ", ".join(importancia.index[::-1][:5]))
# ✅ Debe mostrar: Los 5 más importantes: humedad_relativa_pct, dias_lluvia, dias_desde_fungicida, densidad_follaje, temperatura_media_c

### ✍️ ¿Qué atributos usa más el bosque y cuáles casi no usa? ¿Tiene sentido pensando en cómo aparece un hongo como la botritis?

_Su respuesta:_

### ✍️ 🧠 **De la información al conocimiento.** Escriban la recomendación para la viña: ¿qué modelo conviene usar para decidir qué cuarteles tratar o cosechar antes, qué ganan frente a un árbol solo y qué pierden?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema9.ipynb`, junto con el informe en PDF.